In [4]:
 #STEP 1: Install
# =============================
!pip install transformers scikit-learn torch


In [5]:
# STEP 2: Imports
# =============================
import pandas as pd
import torch
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader


In [6]:
#STEP 3: Upload Dataset
# =============================
from google.colab import files
files.upload()

df = pd.read_csv('IMDB Dataset.csv')

df = df.sample(800, random_state=42)


Saving IMDB Dataset.csv to IMDB Dataset (1).csv


In [7]:
# STEP 4: Preprocessing
# =============================
df = df.dropna()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [8]:
# STEP 5: Split
# =============================
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42
)

val_texts = test_texts[:50]
val_labels = test_labels[:50]

test_texts = test_texts[50:]
test_labels = test_labels[50:]

In [9]:
# STEP 6: Tokenization
# =============================
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN = 32

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx])
        }

train_loader = DataLoader(TextDataset(train_texts, train_labels), batch_size=8, shuffle=True)
val_loader = DataLoader(TextDataset(val_texts, val_labels), batch_size=8)
test_loader = DataLoader(TextDataset(test_texts, test_labels), batch_size=8)


In [10]:
# STEP 7: Model
# =============================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)


Device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# STEP 8: Train
# =============================
def train(model, loader):
    model.train()
    for batch in loader:
        optimizer.zero_grad()
        inputs = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss = model(inputs, attention_mask=mask, labels=labels).loss
        loss.backward()
        optimizer.step()

In [12]:
# STEP 9: Evaluate
# =============================
def evaluate(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            inputs = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(inputs, attention_mask=mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true, preds, average='binary')
    cm = confusion_matrix(true, preds)

    return acc, prec, rec, f1, cm

In [13]:
# STEP 10: Run Training
# =============================
train(model, train_loader)


In [14]:
# STEP 11: Final Evaluation
# =============================
acc, prec, rec, f1, cm = evaluate(model, test_loader)

print("\nResults:")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("Confusion Matrix:\n", cm)


Results:
Accuracy: 0.5727272727272728
Precision: 0.9090909090909091
Recall: 0.17857142857142858
F1: 0.29850746268656714
Confusion Matrix:
 [[53  1]
 [46 10]]


### 📊 Model Analysis

The BERT model was fine-tuned on a subset of the IMDB dataset for sentiment classification.

The model achieved an accuracy of 57%, with high precision (90%) but low recall (17%). This indicates that the model is very conservative in predicting positive reviews. While it makes very few false positive predictions, it fails to identify many actual positive instances.

This behavior is due to limited training data and fewer training epochs. Despite this, BERT demonstrates strong contextual understanding compared to traditional machine learning models.

Overall, increasing the dataset size and training time can improve recall and F1-score.

### 🧪 Experiments

1. Reduced dataset size to speed up training.
2. Used shorter sequence length (32 tokens).
3. Trained for only 1 epoch.

These changes significantly reduced training time but slightly impacted performance.